# Game Analytics: Unlocking Tennis Data with Sportradar API
## Notebook 1 - Configuration & Data Extraction

This notebook handles:
1. **Configuration** - API keys, database connection strings, endpoint URLs
2. **Data Extraction** - Fetching data from 3 Sportradar API endpoints (v3)
3. **JSON Parsing** - Transforming nested JSON into flat relational DataFrames

### Endpoints Used:
- **competitions** to categories + competitions tables
- **complexes** to complexes + venues tables
- **Doubles Competitor Rankings** to competitors + competitor_rankings tables

Note: Sign up at https://console.sportradar.com/signup to get your free API key.
Add the Tennis API trial from https://marketplace.sportradar.com/

**If the API times out**, you can skip extraction and use the pre-fetched `data.sql` file:
```bash
psql -U postgres -d tennis_db -f schema.sql
psql -U postgres -d tennis_db -f data.sql
```

## 1. Import Required Libraries

In [ ]:
import os
import time
import requests
import pandas as pd
import json

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Configuration

Store API keys, endpoint URLs, and database connection details.
Sensitive values are read from environment variables.

**Important:** Sportradar v3 API uses `x-api-key` header for authentication (NOT query param).

In [ ]:
# ---------------------------------------------------------------------------
# Sportradar API Configuration (v3)
# ---------------------------------------------------------------------------
# Set your API key as an environment variable:
#   export SPORTRADAR_API_KEY="your_key_here"
# Or paste it directly below:
API_KEY = os.getenv("SPORTRADAR_API_KEY", "YOUR_API_KEY_HERE")

# Sportradar Tennis API base URL (v3 - current stable version)
# Access level: "trial" (free) or "production" (paid)
ACCESS_LEVEL = "trial"
BASE_URL = f"https://api.sportradar.com/tennis/{ACCESS_LEVEL}/v3"

# Request headers - API key goes in x-api-key header (NOT in URL query param)
HEADERS = {
    "accept": "application/json",
    "x-api-key": API_KEY,
}

# Rate-limit safety: seconds to sleep between consecutive API calls
API_CALL_DELAY = 2.0  # seconds (increased for rate limit safety)

# ---------------------------------------------------------------------------
# API Endpoints (relative to BASE_URL)
# ---------------------------------------------------------------------------
ENDPOINTS = {
    "competitions": f"{BASE_URL}/en/competitions.json",
    "complexes": f"{BASE_URL}/en/complexes.json",
    "doubles_rankings": f"{BASE_URL}/en/double_competitors_rankings.json",
}

# ---------------------------------------------------------------------------
# Database Configuration (PostgreSQL)
# ---------------------------------------------------------------------------
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "your_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "tennis_db")

# SQLAlchemy connection string (PostgreSQL)
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"API Key loaded: {'Yes' if API_KEY != 'YOUR_API_KEY_HERE' else 'No (set SPORTRADAR_API_KEY env var)'}")
print(f"Base URL: {BASE_URL}")
print(f"Database URL: postgresql://{DB_USER}:***@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## 3. Data Extraction Class

The TennisDataExtractor class handles all API calls with retry/back-off for rate limits and timeouts (60s timeout, 5 retries).

In [ ]:
class TennisDataExtractor:
    """Handles all API calls and JSON-to-DataFrame transformation."""

    def __init__(self):
        self.categories_df = pd.DataFrame()
        self.competitions_df = pd.DataFrame()
        self.complexes_df = pd.DataFrame()
        self.venues_df = pd.DataFrame()
        self.competitors_df = pd.DataFrame()
        self.competitor_rankings_df = pd.DataFrame()

    @staticmethod
    def _fetch_data(url, max_retries=5):
        """Fetch JSON from a Sportradar endpoint with retry / back-off."""
        for attempt in range(1, max_retries + 1):
            try:
                # Use 60 second timeout (increased from 30)
                response = requests.get(url, headers=HEADERS, timeout=60)
                if response.status_code == 429:
                    wait = 2 ** attempt
                    print(f"  Rate limited. Waiting {wait}s before retry "
                          f"(attempt {attempt}/{max_retries})...")
                    time.sleep(wait)
                    continue
                response.raise_for_status()
                return response.json()
            except requests.exceptions.HTTPError as e:
                print(f"  HTTP error (attempt {attempt}/{max_retries}): {e}")
            except requests.exceptions.ConnectionError as e:
                print(f"  Connection error (attempt {attempt}/{max_retries}): {e}")
            except requests.exceptions.Timeout:
                print(f"  Timeout (attempt {attempt}/{max_retries})")
            except Exception as e:
                print(f"  Unexpected error (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                wait = 3 * attempt  # 3s, 6s, 9s, 12s back-off
                print(f"  Waiting {wait}s before next retry...")
                time.sleep(wait)
        print("  All retries exhausted for this endpoint.")
        print("  TIP: If the API keeps timing out, use the pre-fetched data.sql file instead:")
        print("       psql -U postgres -d tennis_db -f data.sql")
        return None

    def extract_competitions(self):
        """Fetch competition data and split into categories and competitions DataFrames."""
        print("\n[1/3] Extracting competitions data...")
        data = self._fetch_data(ENDPOINTS["competitions"])
        if not data:
            return
        categories = []
        competitions = []
        comp_list = data.get("competitions", [])
        for comp in comp_list:
            cat = comp.get("category", {})
            categories.append({
                "category_id": cat.get("id"),
                "category_name": cat.get("name"),
            })
            competitions.append({
                "competition_id": comp.get("id"),
                "competition_name": comp.get("name"),
                "parent_id": comp.get("parent_id"),
                "type": comp.get("type"),
                "gender": comp.get("gender"),
                "category_id": cat.get("id"),
            })
        self.categories_df = (
            pd.DataFrame(categories)
            .drop_duplicates(subset=["category_id"])
            .reset_index(drop=True)
        )
        self.competitions_df = pd.DataFrame(competitions)
        print(f"  categories: {len(self.categories_df)} rows")
        print(f"  competitions: {len(self.competitions_df)} rows")

    def extract_complexes(self):
        """Fetch complexes data and split into complexes and venues DataFrames."""
        print("\n[2/3] Extracting complexes data...")
        data = self._fetch_data(ENDPOINTS["complexes"])
        if not data:
            return
        complexes = []
        venues = []
        complex_list = data.get("complexes", [])
        for cx in complex_list:
            complex_id = cx.get("id")
            complexes.append({"complex_id": complex_id, "complex_name": cx.get("name")})
            for venue in cx.get("venues", []):
                city = venue.get("city_name", venue.get("city"))
                if isinstance(city, dict):
                    city = city.get("name")
                country = venue.get("country_name", venue.get("country"))
                if isinstance(country, dict):
                    country = country.get("name")
                country_code = venue.get("country_code")
                if not country_code and isinstance(venue.get("country"), dict):
                    country_code = venue.get("country", {}).get("code")
                venues.append({
                    "venue_id": venue.get("id"),
                    "venue_name": venue.get("name"),
                    "city_name": city,
                    "country_name": country,
                    "country_code": country_code,
                    "timezone": venue.get("time_zone", venue.get("timezone")),
                    "complex_id": complex_id,
                })
        self.complexes_df = pd.DataFrame(complexes).drop_duplicates(subset=["complex_id"]).reset_index(drop=True)
        self.venues_df = pd.DataFrame(venues)
        print(f"  complexes: {len(self.complexes_df)} rows")
        print(f"  venues: {len(self.venues_df)} rows")

    def extract_doubles_rankings(self):
        """Fetch doubles competitor rankings and split into competitors and competitor_rankings."""
        print("\n[3/3] Extracting Doubles Competitor Rankings...")
        data = self._fetch_data(ENDPOINTS["doubles_rankings"])
        if not data:
            return
        competitors = []
        rankings = []
        rank_id = 1
        for ranking_group in data.get("rankings", []):
            for entry in ranking_group.get("competitor_rankings", []):
                comp = entry.get("competitor", {})
                competitor_id = comp.get("id")
                country = comp.get("country")
                country_code = comp.get("country_code")
                competitors.append({
                    "competitor_id": competitor_id,
                    "name": comp.get("name"),
                    "country": country,
                    "country_code": country_code,
                    "abbreviation": comp.get("abbreviation"),
                })
                rankings.append({
                    "rank_id": rank_id,
                    "rank": entry.get("rank"),
                    "movement": entry.get("movement", 0),
                    "points": entry.get("points"),
                    "competitions_played": entry.get("competitions_played", 0),
                    "competitor_id": competitor_id,
                })
                rank_id += 1
        self.competitors_df = pd.DataFrame(competitors).drop_duplicates(subset=["competitor_id"]).reset_index(drop=True)
        self.competitor_rankings_df = pd.DataFrame(rankings)
        print(f"  competitors: {len(self.competitors_df)} rows")
        print(f"  Competitor Rankings: {len(self.competitor_rankings_df)} rows")

    def extract_all(self):
        """Run all three extractions in sequence with rate-limit pauses."""
        self.extract_competitions()
        time.sleep(API_CALL_DELAY)
        self.extract_complexes()
        time.sleep(API_CALL_DELAY)
        self.extract_doubles_rankings()
        return {
            "categories": self.categories_df,
            "competitions": self.competitions_df,
            "complexes": self.complexes_df,
            "venues": self.venues_df,
            "competitors": self.competitors_df,
            "competitor_rankings": self.competitor_rankings_df,
        }

print("TennisDataExtractor class defined.")

## 4. Run Extraction

Now let us extract all data from the three Sportradar API endpoints.

**If you get timeout errors**, your network may be blocking or slowing the connection.
In that case, skip this step and use the pre-fetched `data.sql` file instead:
```bash
psql -U postgres -d tennis_db -f data.sql
```

In [ ]:
extractor = TennisDataExtractor()
data = extractor.extract_all()

## 5. Preview Extracted Data

Let us inspect the first few rows of each DataFrame to verify the extraction.

In [ ]:
for name, df in data.items():
    print(f"\n{'='*60}")
    print(f"  {name.upper()}  ({len(df)} rows, {len(df.columns)} columns)")
    print(f"{'='*60}")
    print(df.head(5))
    print(f"\nColumns: {list(df.columns)}")

## 6. Save DataFrames to CSV (Optional)

Saving intermediate results so the database notebook can load them without re-calling the API.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

for name, df in data.items():
    filepath = f"data/{name}.csv"
    df.to_csv(filepath, index=False)
    print(f"Saved {filepath} ({len(df)} rows)")

print("\nAll DataFrames saved to ./data/")

## Summary

Expected data from the Sportradar Tennis API v3:

| DataFrame | Approx Rows | Source Endpoint |
|-----------|------|-----------------|
| categories | ~18 | competitions |
| competitions | ~6600 | competitions |
| complexes | ~760 | complexes |
| venues | ~4000 | complexes |
| competitors | ~1000 | Doubles Rankings |
| competitor_rankings | ~1000 | Doubles Rankings |

### Next Steps
Open Notebook 2 - Database Setup & Population to create the PostgreSQL schema and insert this data.

### Alternative (if API times out)
Use the pre-fetched `data.sql` file with real API data:
```bash
psql -U postgres -d tennis_db -f schema.sql
psql -U postgres -d tennis_db -f data.sql
```